In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [3]:
flight_weather_df = pd.read_csv('UA2022-2024_flight_weather_data.csv')

In [4]:
flight_weather_df.columns

Index(['FL_DATE', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'CRS_ARR_TIME',
       'ARR_DELAY_NEW', 'DISTANCE', 'YEAR', 'MONTH', 'DAY', 'WEEKDAY',
       'DELAY_CATEGORY', 'ORIGIN_TRAFFIC', 'DEST_TRAFFIC',
       'origin_is_hazardous', 'origin_is_precipitating',
       'origin_is_low_visibility', 'origin_is_thunderstorming',
       'origin_is_severe_wx', 'origin_is_high_wind', 'origin_is_cold',
       'origin_is_hot', 'dest_is_hazardous', 'dest_is_precipitating',
       'dest_is_low_visibility', 'dest_is_thunderstorming',
       'dest_is_severe_wx', 'dest_is_high_wind', 'dest_is_cold',
       'dest_is_hot'],
      dtype='object')

In [5]:
# drop the intermediate columns that are not needed for modeling
flight_weather_df = flight_weather_df.drop(columns=['FL_DATE', 'ARR_DELAY_NEW'])

In [6]:
# check the delay category counts
print(flight_weather_df['DELAY_CATEGORY'].value_counts())
# check the data types of the dataframe
print(flight_weather_df.dtypes)

DELAY_CATEGORY
0    3558459
1     262353
2      13969
Name: count, dtype: int64
ORIGIN                        object
DEST                          object
CRS_DEP_TIME                   int64
CRS_ARR_TIME                   int64
DISTANCE                     float64
YEAR                           int64
MONTH                          int64
DAY                            int64
WEEKDAY                        int64
DELAY_CATEGORY                 int64
ORIGIN_TRAFFIC                 int64
DEST_TRAFFIC                   int64
origin_is_hazardous          float64
origin_is_precipitating      float64
origin_is_low_visibility     float64
origin_is_thunderstorming    float64
origin_is_severe_wx          float64
origin_is_high_wind             bool
origin_is_cold                  bool
origin_is_hot                   bool
dest_is_hazardous            float64
dest_is_precipitating        float64
dest_is_low_visibility       float64
dest_is_thunderstorming      float64
dest_is_severe_wx            flo

In [7]:
# convert every feature that comes after DEST_TRAFFIC to int
for col in flight_weather_df.columns[flight_weather_df.columns.get_loc('DEST_TRAFFIC')+1:]:
    flight_weather_df[col] = flight_weather_df[col].astype(int)
# check the data types of the dataframe again
print(flight_weather_df.dtypes)


ORIGIN                        object
DEST                          object
CRS_DEP_TIME                   int64
CRS_ARR_TIME                   int64
DISTANCE                     float64
YEAR                           int64
MONTH                          int64
DAY                            int64
WEEKDAY                        int64
DELAY_CATEGORY                 int64
ORIGIN_TRAFFIC                 int64
DEST_TRAFFIC                   int64
origin_is_hazardous            int64
origin_is_precipitating        int64
origin_is_low_visibility       int64
origin_is_thunderstorming      int64
origin_is_severe_wx            int64
origin_is_high_wind            int64
origin_is_cold                 int64
origin_is_hot                  int64
dest_is_hazardous              int64
dest_is_precipitating          int64
dest_is_low_visibility         int64
dest_is_thunderstorming        int64
dest_is_severe_wx              int64
dest_is_high_wind              int64
dest_is_cold                   int64
d

3. modeling data preparation

In [8]:
# balance the dataset using undersampling
flight_weather_df_0 = flight_weather_df[flight_weather_df['DELAY_CATEGORY']==0]
flight_weather_df_1 = flight_weather_df[flight_weather_df['DELAY_CATEGORY']==1]
flight_weather_df_2 = flight_weather_df[flight_weather_df['DELAY_CATEGORY']==2]
# sample the 0 category to the size of the 1 category
flight_weather_df_0 = flight_weather_df_0.sample(n=len(flight_weather_df_1))
# sample the 2 category to the size of the 1 category by over sampling
flight_weather_df_2 = flight_weather_df_2.sample(n=len(flight_weather_df_1), replace=True)
# concatenate all 3 categories
flight_weather_df_balanced = pd.concat([flight_weather_df_0, flight_weather_df_1, flight_weather_df_2])
# check the delay category counts
print(flight_weather_df_balanced['DELAY_CATEGORY'].value_counts())

DELAY_CATEGORY
0    262353
1    262353
2    262353
Name: count, dtype: int64


In [9]:
# one-hot encode the categorical columns
flight_weather_df_balanced_encoded = pd.get_dummies(flight_weather_df_balanced, columns=['ORIGIN', 'DEST'], drop_first=True)

In [10]:
# prepare predictor and target variable
X = flight_weather_df_balanced_encoded.drop(columns=['DELAY_CATEGORY'])
y = flight_weather_df_balanced_encoded['DELAY_CATEGORY']
# Convert sparse target (if it exists) to dense format
if hasattr(y, 'toarray'):
    y_dense = y.toarray()  # Convert sparse to dense format
else:
    y_dense = y  # If already dense, no conversion needed

Gradient boosting

In [11]:
# split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier()
gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.55      0.58      0.56     52470
           1       0.53      0.45      0.49     52471
           2       0.59      0.64      0.61     52471

    accuracy                           0.56    157412
   macro avg       0.56      0.56      0.55    157412
weighted avg       0.56      0.56      0.55    157412

